In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.optimize import fsolve, brentq

V0 = 32.0
a = 1.0
m = 1.0
hbar = 1.0


def newton_raphson(f, f_prime, x0, tol=1e-6, max_iter=1000):
    x = x0
    for i in range(max_iter):
        fx = f(x)
        fpx = f_prime(x)

        if not np.isfinite(fx) or not np.isfinite(fpx):
            raise ValueError("Invalid numerical value encountered.")

        if abs(fx) < tol:
            return x

        if abs(fpx) < 1e-12:
            raise ValueError("Derivative is too close to zero.")

        x_new = x - fx / fpx

        if not np.isfinite(x_new) or x_new <= 0 or x_new >= V0:
            raise ValueError("Newton-Raphson left the physical energy range.")

        x = x_new

    raise ValueError("Newton-Raphson did not converge.")


E = sp.Symbol('E', real=True, positive=True)

k_sym = sp.sqrt(2 * m * E) / hbar
kappa_sym = sp.sqrt(2 * m * (V0 - E)) / hbar

even_parity_eq_sym = k_sym * sp.tan(k_sym * a / 2) - kappa_sym
odd_parity_eq_sym = k_sym + kappa_sym * sp.tan(k_sym * a / 2)

even_parity_eq_derivative = sp.diff(even_parity_eq_sym, E)
odd_parity_eq_derivative = sp.diff(odd_parity_eq_sym, E)

even_parity_eq = sp.lambdify(E, even_parity_eq_sym, "numpy")
odd_parity_eq = sp.lambdify(E, odd_parity_eq_sym, "numpy")

even_parity_eq_prime = sp.lambdify(E, even_parity_eq_derivative, "numpy")
odd_parity_eq_prime = sp.lambdify(E, odd_parity_eq_derivative, "numpy")


E_vals = np.linspace(0.01, V0 - 0.01, 1000)

even_vals = even_parity_eq(E_vals)
odd_vals = odd_parity_eq(E_vals)

plt.figure(figsize=(10, 5))
plt.plot(E_vals, even_vals, label="Even Parity", color="blue")
plt.plot(E_vals, odd_vals, label="Odd Parity", color="red")
plt.axhline(0, color="black", linestyle="--")
plt.title("Graphical Solution of Transcendental Equations")
plt.xlabel("Energy E")
plt.ylabel("Function Value")
plt.xlim(0, V0)
plt.ylim(-5, 10)
plt.grid(True)
plt.legend()
plt.show()


def find_roots(func, intervals, points=500):
    roots = []

    for left, right in intervals:
        x_vals = np.linspace(left, right, points)

        for i in range(len(x_vals) - 1):
            x1 = x_vals[i]
            x2 = x_vals[i + 1]

            try:
                y1 = func(x1)
                y2 = func(x2)

                if np.isfinite(y1) and np.isfinite(y2):
                    if y1 * y2 < 0:
                        root = brentq(func, x1, x2)

                        if not roots or all(abs(root - r) > 1e-5 for r in roots):
                            roots.append(root)

            except (ValueError, ZeroDivisionError):
                continue

    return sorted(roots)


pole_energies = [
    ((2*n + 1) * np.pi / a)**2 * hbar**2 / (2 * m)
    for n in range(10)
]

boundaries = [0.001]
boundaries += [p for p in pole_energies if 0 < p < V0]
boundaries += [V0 - 0.001]
boundaries = sorted(boundaries)

intervals = [
    (boundaries[i] + 1e-5, boundaries[i + 1] - 1e-5)
    for i in range(len(boundaries) - 1)
]

even_parity_eigenvalues = find_roots(even_parity_eq, intervals)
odd_parity_eigenvalues = find_roots(odd_parity_eq, intervals)

print("Even parity eigenvalues:")
print(np.round(even_parity_eigenvalues, 4))

print("\nOdd parity eigenvalues:")
print(np.round(odd_parity_eigenvalues, 4))


initial_guesses_even = [3.0, 26.0]
initial_guesses_odd = [12.0]

even_parity_eigenvalues_NR = []
odd_parity_eigenvalues_NR = []

for guess in initial_guesses_even:
    root = newton_raphson(even_parity_eq, even_parity_eq_prime, guess)
    even_parity_eigenvalues_NR.append(root)

for guess in initial_guesses_odd:
    root = newton_raphson(odd_parity_eq, odd_parity_eq_prime, guess)
    odd_parity_eigenvalues_NR.append(root)

print("\nEven parity eigenvalues (Newton-Raphson):")
print(np.round(even_parity_eigenvalues_NR, 4))

print("\nOdd parity eigenvalues (Newton-Raphson):")
print(np.round(odd_parity_eigenvalues_NR, 4))


even_parity_eigenvalues_fsolve = []
odd_parity_eigenvalues_fsolve = []

for guess in even_parity_eigenvalues:
    root = fsolve(even_parity_eq, guess)[0]
    if 0 < root < V0:
        even_parity_eigenvalues_fsolve.append(root)

for guess in odd_parity_eigenvalues:
    root = fsolve(odd_parity_eq, guess)[0]
    if 0 < root < V0:
        odd_parity_eigenvalues_fsolve.append(root)

print("\nEven parity eigenvalues (fsolve):")
print(np.round(even_parity_eigenvalues_fsolve, 4))

print("\nOdd parity eigenvalues (fsolve):")
print(np.round(odd_parity_eigenvalues_fsolve, 4))


def k(E):
    return np.sqrt(2 * m * E) / hbar


def kappa(E):
    return np.sqrt(2 * m * (V0 - E)) / hbar


def even_wavefunction(E, x):
    k_val = k(E)
    kappa_val = kappa(E)

    psi = np.zeros_like(x)
    inside = np.abs(x) <= a / 2

    psi[inside] = np.cos(k_val * x[inside])

    boundary_value = np.cos(k_val * a / 2)

    right = x > a / 2
    left = x < -a / 2

    psi[right] = boundary_value * np.exp(
        -kappa_val * (x[right] - a / 2)
    )

    psi[left] = boundary_value * np.exp(
        -kappa_val * (np.abs(x[left]) - a / 2)
    )

    norm = np.sqrt(np.trapz(np.abs(psi)**2, x))
    return psi / norm


def odd_wavefunction(E, x):
    k_val = k(E)
    kappa_val = kappa(E)

    psi = np.zeros_like(x)
    inside = np.abs(x) <= a / 2

    psi[inside] = np.sin(k_val * x[inside])

    boundary_value = np.sin(k_val * a / 2)

    right = x > a / 2
    left = x < -a / 2

    psi[right] = boundary_value * np.exp(
        -kappa_val * (x[right] - a / 2)
    )

    psi[left] = -boundary_value * np.exp(
        -kappa_val * (np.abs(x[left]) - a / 2)
    )

    norm = np.sqrt(np.trapz(np.abs(psi)**2, x))
    return psi / norm


def potential_well(x, a, V0):
    V = np.zeros_like(x)
    V[np.abs(x) > a / 2] = V0
    return V


x_vals = np.linspace(-2 * a, 2 * a, 1000)
V_vals = potential_well(x_vals, a, V0)

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(
    x_vals,
    even_wavefunction(even_parity_eigenvalues[0], x_vals),
    label=f"Even, E = {even_parity_eigenvalues[0]:.2f}"
)

ax1.plot(
    x_vals,
    odd_wavefunction(odd_parity_eigenvalues[0], x_vals),
    label=f"Odd, E = {odd_parity_eigenvalues[0]:.2f}"
)

if len(even_parity_eigenvalues) > 1:
    ax1.plot(
        x_vals,
        even_wavefunction(even_parity_eigenvalues[1], x_vals),
        label=f"Even, E = {even_parity_eigenvalues[1]:.2f}"
    )

ax2 = ax1.twinx()

ax2.plot(
    x_vals,
    V_vals,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Potential Well"
)

ax1.set_title("Eigenfunctions for Bound States")
ax1.set_xlabel("Position x")
ax1.set_ylabel("Normalized Wavefunction")
ax2.set_ylabel("Potential V(x)")
ax1.set_xlim(-2 * a, 2 * a)
ax1.grid(True)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.show()
